# UNSW-NB15 data audit and split

This notebook reproduces the six-class benchmark filter from the project snapshot. Generic, Reconnaissance, Shellcode, and Worms are excluded because they are outside this benchmark. The official test data is held aside for final evaluation and is not used for split selection or model tuning. Preprocessing that learns parameters, including imputation, encoding, and scaling, belongs inside later training-only model pipelines.

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
RESULTS = ROOT / 'results'
SEED = 42
CLASSES = ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers']
TRAIN_FILE = RAW / 'UNSW_NB15_training-set.csv'
TEST_FILE = RAW / 'UNSW_NB15_testing-set.csv'
outputs = [PROCESSED / 'selected_training.csv', PROCESSED / 'selected_testing.csv', PROCESSED / 'fit_indices.csv', PROCESSED / 'validation_indices.csv', PROCESSED / 'test_indices.csv', RESULTS / 'data_quality_report.json']
assert TRAIN_FILE.is_file() and TEST_FILE.is_file(), 'Required raw training and testing CSV files are missing.'
assert not any(path.exists() for path in outputs), 'An output already exists. Review it before rerunning to avoid overwriting.'
PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
train_raw = pd.read_csv(TRAIN_FILE)
test_raw = pd.read_csv(TEST_FILE)
assert train_raw.columns.tolist() == test_raw.columns.tolist(), 'Train/test schema mismatch.'
assert {'id', 'label', 'attack_cat'}.issubset(train_raw.columns)
predictor_cols = [c for c in train_raw.columns if c not in {'id', 'label', 'attack_cat'}]
assert 'id' not in predictor_cols and not {'label', 'attack_cat'} & set(predictor_cols)
print('Raw shapes:', train_raw.shape, test_raw.shape)
display(pd.DataFrame({'dtype': train_raw.dtypes.astype(str), 'train_missing': train_raw.isna().sum(), 'test_missing': test_raw.isna().sum()}))
display(pd.DataFrame({'training_count': train_raw['attack_cat'].value_counts(dropna=False), 'testing_count': test_raw['attack_cat'].value_counts(dropna=False)}).fillna(0).astype(int))

AssertionError: An output already exists. Review it before rerunning to avoid overwriting.

## Predictor duplicates and conflicting labels
Fingerprints use canonical JSON values from predictor columns only. Repeated fingerprints are grouped for the split. Label conflicts are reported because the same observed predictor vector can carry different target labels.

In [ ]:
def fingerprints(frame):
    return pd.util.hash_pandas_object(frame[predictor_cols], index=False, categorize=True).astype('uint64').map(lambda value: f'{value:016x}')

train_raw['_fingerprint'] = fingerprints(train_raw)
test_raw['_fingerprint'] = fingerprints(test_raw)
def duplicate_summary(frame):
    sizes = frame.groupby('_fingerprint', sort=False).size()
    labels = frame.groupby('_fingerprint', sort=False)['attack_cat'].nunique()
    return {'rows': int(len(frame)), 'unique_predictor_vectors': int(sizes.size), 'rows_beyond_first_duplicate': int(len(frame)-sizes.size), 'duplicate_groups': int((sizes > 1).sum()), 'conflicting_label_groups': int((labels > 1).sum()), 'rows_in_conflicting_groups': int(frame['_fingerprint'].isin(labels[labels > 1].index).sum())}

print('Training duplicate audit:', duplicate_summary(train_raw))
print('Test duplicate audit:', duplicate_summary(test_raw))
display(train_raw.groupby('_fingerprint')['attack_cat'].agg(lambda x: ', '.join(sorted(set(map(str, x))))).loc[lambda x: x.str.contains(',')].head(20).rename('conflicting_labels').to_frame())

Training duplicate audit: {'rows': 175341, 'unique_predictor_vectors': 101040, 'rows_beyond_first_duplicate': 74301, 'duplicate_groups': 9748, 'conflicting_label_groups': 1772, 'rows_in_conflicting_groups': 30528}
Test duplicate audit: {'rows': 82332, 'unique_predictor_vectors': 53946, 'rows_beyond_first_duplicate': 28386, 'duplicate_groups': 6318, 'conflicting_label_groups': 575, 'rows_in_conflicting_groups': 8809}


,conflicting_labels
_fingerprint,
00129e548b1ff3bc,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."
00411989db9c46bd,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."
004a3fdcb1109e0e,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Ge..."
004fae08adb5ee0d,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."
010232df72a32a61,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."
013c0330746c7de5,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."
01500f1afc03a4c1,"Exploits, Fuzzers"
01608a85d63eb96e,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Ge..."
0192a07d54c34e0b,"Analysis, Backdoor, DoS, Exploits, Fuzzers, Re..."


## Six-class filter and grouped split
The official training file is filtered to the six benchmark classes. StratifiedGroupKFold attempts a single approximately 20% validation fold while keeping identical predictor fingerprints together. The official test partition is filtered separately and remains untouched by split selection.

In [ ]:
def select_six(frame):
    return frame.loc[frame['attack_cat'].isin(CLASSES)].copy().reset_index(drop=True)
train = select_six(train_raw)
test = select_six(test_raw)
assert set(train['attack_cat'].unique()) == set(CLASSES)
assert set(test['attack_cat'].unique()) == set(CLASSES)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fit_idx, val_idx = next(splitter.split(train[predictor_cols], train['attack_cat'], groups=train['_fingerprint']))
fit = train.iloc[fit_idx]
validation = train.iloc[val_idx]
fit_groups, val_groups = set(fit['_fingerprint']), set(validation['_fingerprint'])
overlap = fit_groups & val_groups
assert not overlap, f'{len(overlap)} predictor groups cross split'
print('Split sizes:', len(fit), len(validation), 'validation fraction:', round(len(validation)/len(train), 4))
display(pd.crosstab(pd.Series(train['attack_cat'].values[fit_idx], name='fit'), columns='count'))
display(pd.DataFrame({'fit': fit['attack_cat'].value_counts(), 'validation': validation['attack_cat'].value_counts(), 'filtered_train': train['attack_cat'].value_counts()}).reindex(CLASSES).fillna(0).astype(int))
print('Class proportion maximum absolute difference:', float((fit['attack_cat'].value_counts(normalize=True)-validation['attack_cat'].value_counts(normalize=True)).abs().max()))
print('Group overlap:', len(overlap))

Split sizes: 98869 24718 validation fraction: 0.2


col_0,count
fit,
Analysis,1600
Backdoor,1397
DoS,9811
Exploits,26714
Fuzzers,14547
Normal,44800


,fit,validation,filtered_train
attack_cat,,,
Normal,44800,11200,56000
Analysis,1600,400,2000
Backdoor,1397,349,1746
DoS,9811,2453,12264
Exploits,26714,6679,33393
Fuzzers,14547,3637,18184


Class proportion maximum absolute difference: 1.3748832084770068e-05
Group overlap: 0


In [ ]:

# Persist model-ready selected CSVs without the row-order ID or helper fingerprint.
# Index files refer to row positions in selected_training.csv.
processed_columns = [c for c in train.columns if c not in {'id', '_fingerprint'}]
assert 'id' not in processed_columns and '_fingerprint' not in processed_columns
assert {'attack_cat', 'label'}.issubset(processed_columns)
train[processed_columns].to_csv(PROCESSED / 'selected_training.csv', index=False)
test[processed_columns].to_csv(PROCESSED / 'selected_testing.csv', index=False)
pd.DataFrame({'row_index': fit_idx}).to_csv(PROCESSED / 'fit_indices.csv', index=False)
pd.DataFrame({'row_index': val_idx}).to_csv(PROCESSED / 'validation_indices.csv', index=False)
pd.DataFrame({'row_index': np.arange(len(test))}).to_csv(PROCESSED / 'test_indices.csv', index=False)
quality = {
 'raw_shapes': {'training': list(train_raw.shape[:1]) + [len(train_raw.columns)-1], 'testing': list(test_raw.shape[:1]) + [len(test_raw.columns)-1]},
 'schema': {c: str(train_raw[c].dtype) for c in predictor_cols + ['id','label','attack_cat']},
 'missing_values': {'training': train_raw.drop(columns='_fingerprint').isna().sum().astype(int).to_dict(), 'testing': test_raw.drop(columns='_fingerprint').isna().sum().astype(int).to_dict()},
 'raw_class_counts': {'training': train_raw['attack_cat'].value_counts().to_dict(), 'testing': test_raw['attack_cat'].value_counts().to_dict()},
 'duplicate_predictor_audit': {'training': duplicate_summary(train_raw), 'testing': duplicate_summary(test_raw)},
 'benchmark_classes': CLASSES, 'excluded_classes_outside_benchmark': sorted(set(train_raw.attack_cat.dropna()) - set(CLASSES)),
 'filtered_rows': {'training': len(train), 'test': len(test)},
 'filtered_class_counts': {'training': train['attack_cat'].value_counts().reindex(CLASSES, fill_value=0).to_dict(), 'test': test['attack_cat'].value_counts().reindex(CLASSES, fill_value=0).to_dict()},
 'features': {'predictors': predictor_cols, 'excluded_from_model_inputs': ['id','label','attack_cat'], 'id_in_predictors': 'id' in predictor_cols, 'processed_columns': processed_columns, 'processed_column_count': len(processed_columns), 'excluded_from_processed_csvs': ['id','_fingerprint']},
 'split': {'method': 'StratifiedGroupKFold', 'folds': 5, 'random_state': SEED, 'fit_rows': len(fit_idx), 'validation_rows': len(val_idx), 'validation_fraction': len(val_idx)/len(train), 'group_overlap_count': len(overlap), 'fit_class_counts': fit['attack_cat'].value_counts().reindex(CLASSES, fill_value=0).to_dict(), 'validation_class_counts': validation['attack_cat'].value_counts().reindex(CLASSES, fill_value=0).to_dict(), 'class_proportion_max_abs_difference': float((fit['attack_cat'].value_counts(normalize=True)-validation['attack_cat'].value_counts(normalize=True)).abs().max())},
 'test_usage': 'filtered and saved separately; not used for split selection or model tuning',
 'preprocessing': 'no imputation, encoding, or scaling fitted in this notebook'
}
(RESULTS / 'data_quality_report.json').write_text(json.dumps(quality, indent=2, default=int), encoding='utf-8')
# Lightweight persistence checks
assert all(p.is_file() for p in outputs)
assert len(fit_idx)+len(val_idx) == len(train)
assert 'id' not in predictor_cols and not overlap
assert all('id' not in pd.read_csv(path, nrows=0).columns and '_fingerprint' not in pd.read_csv(path, nrows=0).columns for path in outputs[:2])
print('Saved outputs:', *[str(p.relative_to(ROOT)) for p in outputs], sep='\n- ')
print('All lightweight checks passed.')

Saved outputs:
- data\processed\selected_training.csv
- data\processed\selected_testing.csv
- data\processed\fit_indices.csv
- data\processed\validation_indices.csv
- data\processed\test_indices.csv
- results\data_quality_report.json
All lightweight checks passed.
